# One-test-tube salt-analysis validation

Upload both independent images before running. The sequence is dry salt -> dissolve -> acidified solution -> silver nitrate precipitate -> ammonia dissolution. Only the final transition is chained.


In [ ]:
import json, os, subprocess
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/kaggle/working")
REPOS = {
    "Wan2.2": ("https://github.com/abhinavk0006/Wan2.2.git", "cfae035"),
    "image-to-video-pipeline": ("https://github.com/abhinavk0006/image-to-video-pipeline.git", "acb6103"),
    "Edu-video-gen-dataset": ("https://github.com/abhinavk0006/Edu-video-gen-dataset.git", "53b5222"),
}
for name, (url, commit) in REPOS.items():
    path = ROOT / name
    if not path.exists():
        subprocess.run(["git", "clone", url, str(path)], check=True)
    subprocess.run(["git", "fetch", "origin", "main"], cwd=path, check=True)
    subprocess.run(["git", "checkout", "--detach", commit], cwd=path, check=True)
dry = list(Path("/kaggle/input").rglob("salt_analysis_one_test_tube_dry_salt.jpeg"))
acid = list(Path("/kaggle/input").rglob("salt_analysis_one_test_tube_acidified_solution.jpeg"))
assert len(dry) == 1, "Add exactly one dry-salt image with the required filename"
assert len(acid) == 1, "Add exactly one acidified-solution image with the required filename"
DRY_IMAGE, ACID_IMAGE = dry[0], acid[0]
print("Dry reference:", DRY_IMAGE)
print("Acidified-solution reference:", ACID_IMAGE)


In [ ]:
pipeline = ROOT / "image-to-video-pipeline"
fixture_dir = pipeline / "kaggle_experiments"
fixture_dir.mkdir(exist_ok=True)
negative = "extra test tubes, duplicate vessels, multiple droppers, crowded rack, unrelated apparatus, blurry, warped glass, text, watermark, camera shake"
def clip(name, image, before, action, after, procedure, observation, motion, chained=False):
    data = {"name": name, "input_frame_path": str(image) if image else None, "state_before": before, "action": action, "state_after": after, "visual_priority": "high", "continuity_required": chained, "reference_mode": "chained" if chained else "independent", "needs_reference_image": not chained, "duration_seconds": 2.0, "procedure_text": procedure, "observation_text": observation, "prompt_bundle": {"motion_prompt": motion, "negative_prompt": negative}}
    if chained:
        data.update({"reference_policy": "previous_state_keyframe", "reference_clip": "silver_nitrate_chloride_test", "handoff_policy": "final", "handoff_entity": "white silver chloride precipitate"})
    else:
        data.update({"reference_policy": "state_keyframe"})
    return data
clips = [
  clip("prepare_unknown_salt", DRY_IMAGE, "one dry test tube with a small amount of white unknown salt", "add water and swirl gently until the salt dissolves", "clear unknown salt solution", "Add distilled water to the dry unknown salt and dissolve it.", "The white salt dissolves to form a clear solution.", "Use exactly one test tube containing dry white salt. Add a small amount of water and swirl gently until the salt dissolves. Do not create another vessel."),
  clip("silver_nitrate_chloride_test", ACID_IMAGE, "one test tube containing clear acidified unknown salt solution", "add dilute silver nitrate slowly and gently swirl once", "one thick white curdy silver chloride precipitate remains visible in the same test tube", "Add dilute silver nitrate solution to the acidified salt solution.", "A thick white curdy precipitate of silver chloride forms.", "Use exactly one test tube. Add dilute silver nitrate slowly. A single thick white curdy precipitate forms and remains visible in that same tube. Keep the camera fixed and create no second vessel."),
  clip("ammonia_dissolves_chloride_ppt", None, "the same one test tube with thick white curdy silver chloride precipitate already visible", "add dilute ammonia gradually to the existing precipitate", "the white precipitate dissolves, leaving one clear colorless solution", "Add dilute ammonia solution to the silver chloride precipitate.", "The white precipitate dissolves in ammonia and the solution becomes clear.", "Continue with the same single test tube and visible white precipitate. Add dilute ammonia gradually. The precipitate dissolves in place, leaving one clear colorless solution. Do not introduce another tube.", chained=True)
]
fixture = {"name": "One-Tube Chloride Salt Analysis", "subject": "Qualitative salt analysis", "educational_goal": "Prepare one salt solution, confirm chloride using silver nitrate, and verify dissolution in ammonia.", "clips": clips}
fixture_path = fixture_dir / "salt_analysis_one_tube.json"
fixture_path.write_text(json.dumps(fixture, indent=2), encoding="utf-8")
print(fixture_path)


## Generate with the persistent worker

Use a fresh output directory. The fourth clip uses the third clip's explicit final handoff.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
OUT = "/kaggle/working/wan_salt_analysis_one_tube"
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/salt_analysis_one_tube.json --initial-image $DRY_IMAGE --output-dir $OUT --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode chain --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=6 --generator-arg=--gpu1-memory-gib=10 --generator-arg=--memory-telemetry


In [ ]:
from pathlib import Path
out = Path("/kaggle/working/wan_salt_analysis_one_tube/one_tube_chloride_salt_analysis")
for path in sorted(out.rglob("*.mp4")):
    print(path, path.stat().st_size, "bytes")
print("Final:", out / "final_video.mp4")
